# TFM · Fase 2: construcción de backlogs de prueba en Jira
Genera los tres escenarios de prueba, los carga en Jira Cloud con la API y comprueba que lo cargado coincide con el origen.

| Escenario | Contenido | Origen |
|---|---|---|
| **E1** | Backlog pequeño (15 historias, pocas dependencias): biblioteca universitaria | Definido a mano |
| **E2** | Backlog mediano (30 historias, 139 SP, 28 con dependencias): tienda en línea | Definido a mano |
| **E3** | Backlog extenso (80 historias), sintético | Generado con semilla fija |

### Preparación en Jira (una sola vez, por proyecto)
1. Crea un **proyecto Scrum** para cada escenario (p. ej. `BIB` para E1, `TDA` para E2 y `SIN` para E3).
2. Agregar el campo numérico **`Valor de negocio`** al tipo *Historia / Story*.
3. Comprobar que **story points** esté habilitado (*Story point estimate* o *Story Points*).
4. El tipo de vínculo **Blocks** existe por defecto en Jira Cloud.

La celda 4 comprueba todo esto antes de cargar.

**Secrets de Colab** (🔑, con *Notebook access* activado): `JIRA_URL`, `JIRA_EMAIL`, `JIRA_API_TOKEN`.

## 1. Librerías

In [ ]:
import csv, io, json, math, os, random, sys, time, platform
import requests
import pandas as pd
from getpass import getpass

os.makedirs("escenarios", exist_ok=True)
print("Python", platform.python_version(), "| requests", requests.__version__, "| pandas", pd.__version__)

Python 3.13.18 | requests 2.32.4 | pandas 2.2.3


## 2. Credenciales y configuración
- `proyecto`: clave del proyecto Jira de cada escenario.
- `cargar`: `False` para no volver a cargar un escenario.

In [ ]:
def leer_config(nombre, secreto=False):
    try:
        from google.colab import userdata
        valor = userdata.get(nombre)
        if valor:
            return valor
    except Exception:
        pass
    return getpass(f"{nombre}: ") if secreto else input(f"{nombre}: ")

JIRA_URL       = leer_config("JIRA_URL").rstrip("/")
JIRA_EMAIL     = leer_config("JIRA_EMAIL")
JIRA_API_TOKEN = leer_config("JIRA_API_TOKEN", secreto=True)

ESCENARIOS = {
    "E1": {"proyecto": "BIB", "cargar": False},
    "E2": {"proyecto": "TDA", "cargar": True},
    "E3": {"proyecto": "SIN", "cargar": False},
}
SEMILLA   = 42    # semilla fija de E3 (reproducibilidad)
N_E3      = 80    # historias de E3
VELOCIDAD = 25    # SP por sprint (diseño experimental)

print("Sitio:", JIRA_URL)
for e, c in ESCENARIOS.items():
    print(f"  {e}: proyecto {c['proyecto']} | cargar={c['cargar']}")

Sitio: https://comunidadunir-team-uuwgkxpf.atlassian.net
  E1: proyecto BIB | cargar=False
  E2: proyecto TDA | cargar=True
  E3: proyecto SIN | cargar=False


## 3. Cliente de Jira Cloud

In [ ]:
NOMBRES_STORY_POINTS = ["Story point estimate", "Story Points",
                        "Estimación de puntos de historia", "Puntos de historia"]
NOMBRE_VALOR_NEGOCIO = "Valor de negocio"
PRIORIDAD_NUM = {"Highest": 5, "High": 4, "Medium": 3, "Low": 2, "Lowest": 1}


class Jira:
    def __init__(self, url, email, token):
        self.base = url.rstrip("/")
        self.s = requests.Session()
        self.s.auth = (email, token)
        self.s.headers.update({"Accept": "application/json", "Content-Type": "application/json"})

    def _req(self, metodo, ruta, **kw):
        for intento in range(5):
            r = self.s.request(metodo, f"{self.base}{ruta}", timeout=30, **kw)
            if r.status_code == 429:
                time.sleep(int(r.headers.get("Retry-After", 5)))
                continue
            if r.status_code >= 400:
                raise RuntimeError(f"{metodo} {ruta} -> {r.status_code}: {r.text[:500]}")
            return r.json() if r.content else None
        raise RuntimeError(f"{metodo} {ruta}: demasiados reintentos")

    def get(self, ruta, **kw):    return self._req("GET", ruta, **kw)
    def post(self, ruta, json):   return self._req("POST", ruta, json=json)
    def put(self, ruta, json):    return self._req("PUT", ruta, json=json)
    def delete(self, ruta):       return self._req("DELETE", ruta)

    def campos(self):
        por_nombre = {}
        for f in self.get("/rest/api/3/field"):
            por_nombre.setdefault(f["name"], []).append(f["id"])
        return por_nombre

    def buscar(self, jql, campos):
        issues, token = [], None
        while True:
            cuerpo = {"jql": jql, "fields": campos, "maxResults": 100}
            if token:
                cuerpo["nextPageToken"] = token
            r = self.post("/rest/api/3/search/jql", cuerpo)
            issues += r.get("issues", [])
            token = r.get("nextPageToken")
            if r.get("isLast", True) or not token:
                return issues


def adf(texto):
    """Texto plano a Atlassian Document Format (requerido por la API v3)."""
    return {"type": "doc", "version": 1,
            "content": [{"type": "paragraph", "content": [{"type": "text", "text": linea}]}
                        for linea in texto.splitlines() if linea.strip()]}


def _lista(r, *claves):
    for k in claves:
        if isinstance(r, dict) and k in r:
            return r[k]
    return r if isinstance(r, list) else []


jira = Jira(JIRA_URL, JIRA_EMAIL, JIRA_API_TOKEN)
print("Conectado como:", jira.get("/rest/api/3/myself")["displayName"])

Conectado como: Tracy Rocío Cárdenas Arichábala


## 4. Verificar la preparación de Jira

In [ ]:
def verificar_proyecto(clave):
    r = {"proyecto": clave}
    try:
        p = jira.get(f"/rest/api/3/project/{clave}")
        r["existe"] = f"✔ {p['name']}"
    except RuntimeError:
        r["existe"] = "✘ no existe o sin permisos"
        return r, None

    # Los espacios gestionados por el equipo devuelven tableros tipo "simple": se comprueba si admiten sprints
    tableros = jira.get("/rest/agile/1.0/board", params={"projectKeyOrId": clave}).get("values", [])
    r["tablero_scrum"] = "✘ sin tablero"
    for t in tableros:
        if t["type"] == "scrum":
            r["tablero_scrum"] = "✔"
            break
        try:
            jira.get(f"/rest/agile/1.0/board/{t['id']}/sprint", params={"maxResults": 1})
            r["tablero_scrum"] = "✔ (con sprints)"
            break
        except RuntimeError:
            r["tablero_scrum"] = "✘ sin sprints (activa Sprints en Funciones)"

    tipos = _lista(jira.get(f"/rest/api/3/issue/createmeta/{clave}/issuetypes"), "issueTypes", "values")
    historia = next((t for t in tipos if t["name"] in ("Story", "Historia")), None)
    if not historia:
        r["tipo_historia"] = "✘ no hay tipo Story/Historia"
        return r, None
    r["tipo_historia"] = f"✔ {historia['name']}"

    campos = _lista(jira.get(f"/rest/api/3/issue/createmeta/{clave}/issuetypes/{historia['id']}"),
                    "fields", "values", "results")
    por_nombre = {c["name"]: c["fieldId"] for c in campos}
    sp = next((por_nombre[n] for n in NOMBRES_STORY_POINTS if n in por_nombre), None)
    vn = por_nombre.get(NOMBRE_VALOR_NEGOCIO)
    r["story_points"] = f"✔ {sp}" if sp else "✘ no está en la pantalla"
    if vn:
        r["valor_negocio"] = f"✔ {vn}"
    else:
        numericos = [c["name"] for c in campos if c.get("schema", {}).get("type") == "number"
                     and c["fieldId"] != sp]
        r["valor_negocio"] = "✘ no está en la pantalla" + (f" (numéricos: {', '.join(numericos)})" if numericos else "")
    r["prioridad"] = "✔" if "priority" in por_nombre.values() else "✘ agrega el campo Prioridad"
    return r, {"tipo": historia["name"], "story_points": sp, "valor_negocio": vn,
               "prioridad": "priority" in por_nombre.values()}


tipos_vinculo = [t["name"] for t in jira.get("/rest/api/3/issueLinkType")["issueLinkTypes"]]
print("Vínculo Blocks:", "✔" if "Blocks" in tipos_vinculo else "✘ no existe")

META, filas_ver = {}, []
for clave in sorted({c["proyecto"] for c in ESCENARIOS.values()}):
    r, meta = verificar_proyecto(clave)
    filas_ver.append(r)
    if meta:
        META[clave] = meta
display(pd.DataFrame(filas_ver).fillna(""))

Vínculo Blocks: ✔


,proyecto,existe,tablero_scrum,tipo_historia,story_points,valor_negocio,prioridad
0,BIB,✔ BIB,✔ (con sprints),✔ Historia,✔ customfield_10016,✔ customfield_10079,✔
1,SIN,✔ SIN,✔ (con sprints),✔ Historia,✔ customfield_10016,✔ customfield_10080,✔
2,TDA,✔ TDA,✔ (con sprints),✔ Historia,✔ customfield_10016,✔ customfield_10045,✔


## 5. Definición de los escenarios E1 y E2

In [ ]:
E1_CSV = """Issue Id,Summary,Issue Type,Description,Priority,Story Points,Valor de negocio,Labels,Depende de
1,Registro de socios de la biblioteca,Story,"Como estudiante, quiero registrarme como socio con mi correo institucional para poder usar los servicios de la biblioteca.",Highest,5,80,socios,
2,Inicio de sesión de socios,Story,"Como socio, quiero iniciar sesión con mis credenciales para acceder a mis reservas y préstamos.",Highest,3,85,socios,1
3,Catálogo de libros,Story,"Como bibliotecario, quiero registrar los libros con título, autor, ISBN y ubicación para mantener el catálogo disponible en línea.",Highest,5,90,catalogo,
4,Búsqueda por título y autor,Story,"Como socio, quiero buscar libros por título o autor para encontrar rápidamente lo que necesito.",High,3,70,catalogo,3
5,Reserva de libros en línea,Story,"Como socio, quiero reservar un libro disponible para retirarlo después en el mostrador.",Highest,8,95,prestamos,"2,3"
6,Registro de préstamos en mostrador,Story,"Como bibliotecario, quiero registrar el préstamo de un ejemplar a un socio para controlar quién tiene cada libro.",High,5,75,prestamos,
7,Registro de devoluciones,Story,"Como bibliotecario, quiero registrar la devolución de un ejemplar para que vuelva a estar disponible.",High,3,65,prestamos,
8,Aviso de vencimiento por correo,Story,"Como socio, quiero recibir un correo antes de que venza mi préstamo para devolver el libro a tiempo.",Medium,3,50,notificaciones,6
9,Cálculo de multas por retraso,Story,"Como bibliotecario, quiero que el sistema calcule la multa por días de retraso para aplicarla de forma uniforme.",Medium,5,45,prestamos,
10,Historial de préstamos del socio,Story,"Como socio, quiero consultar mis préstamos anteriores para recordar qué libros he leído.",Medium,2,40,socios,
11,Lista de espera de libros prestados,Story,"Como socio, quiero anotarme en una lista de espera cuando un libro no está disponible para que me avisen al devolverse.",Low,5,35,prestamos,5
12,Reporte de libros más prestados,Story,"Como bibliotecario, quiero ver los libros más prestados por período para decidir nuevas adquisiciones.",Low,3,30,reportes,
13,Alta y baja de ejemplares,Story,"Como bibliotecario, quiero registrar nuevos ejemplares y dar de baja los dañados para mantener el inventario real.",Medium,3,55,catalogo,
14,Recomendaciones de lectura,Story,"Como socio, quiero recibir recomendaciones según mis préstamos anteriores para descubrir libros de mi interés.",Lowest,8,20,catalogo,
15,Exportación del catálogo a CSV,Story,"Como bibliotecario, quiero exportar el catálogo a un archivo CSV para compartirlo con otras dependencias.",Lowest,2,15,reportes,
"""

E2_CSV = """Issue Id,Summary,Issue Type,Description,Priority,Story Points,Valor de negocio,Labels,Depende de
1,Registro de usuarios con correo electrónico,Story,"Como cliente, quiero crear una cuenta con mi correo y contraseña para poder comprar y dar seguimiento a mis pedidos.",Highest,5,90,usuarios,
2,Inicio de sesión,Story,"Como cliente registrado, quiero iniciar sesión con mis credenciales para acceder a mi cuenta de forma segura.",Highest,3,90,usuarios,1
3,Recuperación de contraseña,Story,"Como cliente registrado, quiero restablecer mi contraseña mediante un enlace enviado a mi correo para recuperar el acceso si la olvido.",Medium,3,50,usuarios,2
4,Gestión del perfil de usuario,Story,"Como cliente registrado, quiero editar mis datos personales y direcciones para mantener mi información actualizada.",Low,3,35,usuarios,2
5,Modelo de datos de productos y categorías,Story,"Como administrador, quiero contar con una estructura de productos y categorías para organizar el catálogo de la tienda.",Highest,5,85,catalogo,
6,Listado de productos por categoría,Story,"Como cliente, quiero ver los productos agrupados por categoría para explorar el catálogo fácilmente.",High,5,80,catalogo,5
7,Búsqueda de productos por nombre,Story,"Como cliente, quiero buscar productos por nombre o palabra clave para encontrar rápidamente lo que necesito.",High,5,75,catalogo,6
8,Filtros por precio y categoría,Story,"Como cliente, quiero filtrar los resultados por rango de precio y categoría para acotar la búsqueda.",Medium,3,55,catalogo,7
9,Detalle de producto con imágenes,Story,"Como cliente, quiero ver la descripción, precio e imágenes de un producto para decidir si lo compro.",High,3,70,catalogo,6
10,Reseñas y calificaciones de productos,Story,"Como cliente registrado, quiero calificar y comentar productos que compré para ayudar a otros compradores.",Low,5,30,catalogo,"9,2"
11,Agregar producto al carrito,Story,"Como cliente, quiero añadir productos al carrito de compras para comprar varios productos a la vez.",Highest,5,95,carrito,9
12,Modificar cantidades y eliminar productos del carrito,Story,"Como cliente, quiero cambiar cantidades o quitar productos del carrito para ajustar mi compra antes de pagar.",High,3,70,carrito,11
13,Carrito persistente para usuarios registrados,Story,"Como cliente registrado, quiero que mi carrito se conserve entre sesiones para continuar mi compra más tarde.",Medium,5,45,carrito,"11,2"
14,Cálculo de costos de envío,Story,"Como cliente, quiero conocer el costo de envío según mi ciudad para saber el total antes de pagar.",High,5,65,carrito,11
15,Checkout: dirección y datos de envío,Story,"Como cliente registrado, quiero ingresar la dirección y el método de envío para recibir mi pedido correctamente.",Highest,5,90,pagos,"12,2"
16,Integración con pasarela de pago,Story,"Como cliente registrado, quiero pagar con tarjeta de crédito o débito para completar mi compra en línea.",Highest,13,100,pagos,15
17,Confirmación de pedido por correo,Story,"Como cliente, quiero recibir un correo con el resumen de mi pedido para tener constancia de la compra.",High,3,60,pedidos,16
18,Historial de pedidos del cliente,Story,"Como cliente registrado, quiero consultar mis pedidos anteriores para revisar mis compras.",Medium,3,50,pedidos,"16,2"
19,Seguimiento del estado del pedido,Story,"Como cliente registrado, quiero ver el estado actual de mi pedido para saber cuándo llegará.",Medium,5,55,pedidos,18
20,Cancelación de pedido,Story,"Como cliente registrado, quiero cancelar un pedido que aún no se ha enviado para corregir compras por error.",Low,3,35,pedidos,18
21,Panel de administración: gestión de productos,Story,"Como administrador, quiero crear, editar y eliminar productos para mantener el catálogo actualizado.",High,8,75,administracion,5
22,Gestión de inventario y stock,Story,"Como administrador, quiero registrar y actualizar el stock de cada producto para evitar vender productos agotados.",High,5,70,administracion,21
23,Alertas de stock bajo,Story,"Como administrador, quiero recibir una alerta cuando un producto tenga poco stock para reabastecerlo a tiempo.",Low,2,30,administracion,22
24,Gestión de pedidos desde el panel,Story,"Como administrador, quiero ver y cambiar el estado de los pedidos para coordinar los despachos.",Medium,5,55,administracion,"16,21"
25,Cupones de descuento,Story,"Como cliente, quiero aplicar un cupón de descuento en el checkout para aprovechar promociones.",Low,5,40,pagos,15
26,Reporte de ventas por período,Story,"Como administrador, quiero generar un reporte de ventas por rango de fechas para analizar el desempeño de la tienda.",Medium,5,45,reportes,24
27,Lista de deseos,Story,"Como cliente registrado, quiero guardar productos en una lista de deseos para comprarlos más adelante.",Lowest,3,20,catalogo,"9,2"
28,Productos recomendados,Story,"Como cliente, quiero ver productos relacionados con el que estoy viendo para descubrir otros productos de interés.",Lowest,8,25,catalogo,9
29,Diseño adaptable para dispositivos móviles,Story,"Como cliente, quiero usar la tienda cómodamente desde mi celular para comprar desde cualquier lugar.",Medium,5,50,frontend,6
30,Registro de auditoría de acciones administrativas,Story,"Como administrador, quiero consultar quién modificó productos y pedidos para garantizar la trazabilidad.",Lowest,3,20,administracion,21
"""

## 6. Generador del escenario sintético E3
Reglas del diseño:
- Story points tomados de la escala de Fibonacci (1, 2, 3, 5, 8, 13).
- Valor de negocio entero entre 1 y 100.
- La prioridad se deriva del valor, con un 25 % de ruido, para que prioridad y valor no sean idénticos.
- Con probabilidad 0,5 una historia depende de 1 a 3 historias **anteriores** dentro de una ventana de 15. Como las dependencias solo apuntan hacia atrás, el grafo no puede tener ciclos.

Usa `random.Random(SEMILLA)` de la librería estándar: la misma semilla produce siempre el mismo backlog.

In [ ]:
FIBONACCI = [1, 2, 3, 5, 8, 13]
PESOS_SP = [0.10, 0.15, 0.25, 0.25, 0.15, 0.10]
MODULOS = {
    "usuarios":       ("usuario",        ["cuentas de usuario", "roles y permisos", "perfil del usuario", "autenticación en dos pasos", "sesiones activas"]),
    "catalogo":       ("cliente",        ["fichas de producto", "categorías", "atributos de producto", "imágenes del catálogo", "precios especiales"]),
    "inventario":     ("bodeguero",      ["existencias", "movimientos de bodega", "proveedores", "órdenes de compra", "lotes"]),
    "ventas":         ("vendedor",       ["cotizaciones", "pedidos de venta", "descuentos", "clientes corporativos", "metas de venta"]),
    "pagos":          ("cliente",        ["pagos con tarjeta", "facturas electrónicas", "notas de crédito", "reembolsos", "conciliación bancaria"]),
    "logistica":      ("coordinador",    ["despachos", "rutas de entrega", "guías de remisión", "transportistas", "devoluciones"]),
    "reportes":       ("gerente",        ["indicadores de ventas", "reportes de inventario", "tableros de gestión", "reportes financieros", "auditoría de cambios"]),
    "notificaciones": ("administrador",  ["alertas por correo", "plantillas de mensajes", "notificaciones push", "recordatorios", "preferencias de aviso"]),
}
ACCIONES = [("Registrar", "registrar"), ("Consultar", "consultar"), ("Editar", "editar"),
            ("Exportar", "exportar"), ("Validar", "validar"), ("Aprobar", "aprobar"),
            ("Filtrar", "filtrar"), ("Importar", "importar")]
PRIORIDADES = ["Lowest", "Low", "Medium", "High", "Highest"]


def prioridad_desde_valor(valor, rng):
    nivel = min(4, (valor - 1) // 20)          # 1-20 Lowest ... 81-100 Highest
    if rng.random() < 0.25:                    # ruido: la prioridad no es un espejo del valor
        nivel = max(0, min(4, nivel + rng.choice([-1, 1])))
    return PRIORIDADES[nivel]


def generar_e3(n=80, semilla=42, prob_dependencia=0.5, max_deps=3, ventana=15):
    rng = random.Random(semilla)
    combinaciones = [(mod, rol, obj, a) for mod, (rol, objs) in MODULOS.items()
                     for obj in objs for a in ACCIONES]
    elegidas = rng.sample(combinaciones, n)
    filas = []
    for i, (mod, rol, obj, (verbo, infinitivo)) in enumerate(elegidas, start=1):
        sp = rng.choices(FIBONACCI, weights=PESOS_SP)[0]
        valor = rng.randint(1, 100)
        deps = []
        if i > 1 and rng.random() < prob_dependencia:
            candidatos = list(range(max(1, i - ventana), i))      # solo historias anteriores → sin ciclos
            deps = sorted(rng.sample(candidatos, min(len(candidatos), rng.randint(1, max_deps))))
        filas.append({
            "Issue Id": str(i),
            "Summary": f"{verbo} {obj}",
            "Issue Type": "Story",
            "Description": f"Como {rol}, quiero {infinitivo} {obj} para mejorar la gestión del módulo de {mod}.",
            "Priority": prioridad_desde_valor(valor, rng),
            "Story Points": str(sp),
            "Valor de negocio": str(valor),
            "Labels": mod,
            "Depende de": ",".join(map(str, deps)),
        })
    return filas

## 7. Construir, validar y guardar los escenarios

In [ ]:
def leer_csv_texto(texto):
    return list(csv.DictReader(io.StringIO(texto)))


def deps(fila):
    return [d.strip() for d in fila["Depende de"].split(",") if d.strip()]


def validar_escenario(filas):
    ids = [f["Issue Id"] for f in filas]
    problemas = []
    if len(set(ids)) != len(ids): problemas.append("Ids repetidos")
    res = [f["Summary"] for f in filas]
    if len(set(res)) != len(res): problemas.append("Resúmenes repetidos")
    pos = {x: k for k, x in enumerate(ids)}
    for f in filas:
        for d in deps(f):
            if d not in pos: problemas.append(f"{f['Issue Id']} depende de {d}, inexistente")
    grafo = {f["Issue Id"]: deps(f) for f in filas}
    estado = {}
    def visitar(n):
        estado[n] = 1
        for m in grafo.get(n, []):
            if estado.get(m) == 1: problemas.append(f"Ciclo en {n} -> {m}")
            elif m not in estado and m in grafo: visitar(m)
        estado[n] = 2
    for n in grafo:
        if n not in estado: visitar(n)
    return problemas


def resumen_escenario(nombre, filas, velocidad=25):
    sp = sum(float(f["Story Points"]) for f in filas)
    vn = sum(float(f["Valor de negocio"]) for f in filas)
    con_dep = sum(1 for f in filas if deps(f))
    vinc = sum(len(deps(f)) for f in filas)
    smin = math.ceil(sp / velocidad)
    return {"escenario": nombre, "historias": len(filas), "story_points": sp, "valor_total": vn,
            "con_dependencias": con_dep, "vinculos_blocks": vinc,
            "sp_medio": round(sp / len(filas), 2), "sprints_min": smin, "horizonte": smin + 1}


BACKLOGS = {
    "E1": leer_csv_texto(E1_CSV),
    "E2": leer_csv_texto(E2_CSV),
    "E3": generar_e3(n=N_E3, semilla=SEMILLA),
}

resumenes = []
for nombre, filas in BACKLOGS.items():
    problemas = validar_escenario(filas)
    print(f"{nombre}: {'válido ✔' if not problemas else problemas}")
    with open(f"escenarios/{nombre}.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(filas[0].keys()))
        w.writeheader()
        w.writerows(filas)
    resumenes.append(resumen_escenario(nombre, filas, VELOCIDAD))

tabla_escenarios = pd.DataFrame(resumenes)
tabla_escenarios.to_csv("escenarios/resumen_escenarios.csv", index=False, encoding="utf-8-sig")
display(tabla_escenarios)
print("Archivos en escenarios/: E1.csv, E2.csv, E3.csv, resumen_escenarios.csv")

E1: válido ✔
E2: válido ✔
E3: válido ✔


,escenario,historias,story_points,valor_total,con_dependencias,vinculos_blocks,sp_medio,sprints_min,horizonte
0,E1,15,63.0,850.0,5,6,4.20,3,4
1,E2,30,139.0,1755.0,28,34,4.63,6,7
2,E3,80,383.0,4139.0,50,98,4.79,16,17


Archivos en escenarios/: E1.csv, E2.csv, E3.csv, resumen_escenarios.csv


## 8. Funciones de carga

In [ ]:
def etiqueta(escenario):
    return f"escenario-{escenario}"


def existentes(proyecto, escenario):
    return jira.buscar(f'project = "{proyecto}" AND labels = "{etiqueta(escenario)}"', ["summary"])


def crear_historia(proyecto, escenario, fila, meta):
    fields = {
        "project": {"key": proyecto},
        "issuetype": {"name": meta["tipo"]},
        "summary": fila["Summary"],
        "description": adf(fila["Description"]),
        "labels": [l for l in (fila["Labels"], etiqueta(escenario)) if l],
    }
    if meta["prioridad"]:
        fields["priority"] = {"name": fila["Priority"]}
    if meta["story_points"]:
        fields[meta["story_points"]] = float(fila["Story Points"])
    if meta["valor_negocio"]:
        fields[meta["valor_negocio"]] = float(fila["Valor de negocio"])
    try:
        return jira.post("/rest/api/3/issue", {"fields": fields})["key"]
    except RuntimeError as e:
        if "priority" not in str(e):
            raise
        print(f"  Aviso: prioridad '{fila['Priority']}' no aceptada; se crea sin prioridad.")
        fields.pop("priority", None)
        return jira.post("/rest/api/3/issue", {"fields": fields})["key"]


INVERTIR = None   # se determina con el primer vínculo creado


def _vincular(bloqueante, dependiente, invertir):
    a, b = (dependiente, bloqueante) if invertir else (bloqueante, dependiente)
    jira.post("/rest/api/3/issueLink",
              {"type": {"name": "Blocks"}, "outwardIssue": {"key": a}, "inwardIssue": {"key": b}})


def _direccion_correcta(bloqueante, dependiente):
    """True si 'dependiente' aparece como 'is blocked by' 'bloqueante'."""
    links = jira.get(f"/rest/api/3/issue/{dependiente}", params={"fields": "issuelinks"})["fields"]["issuelinks"]
    for l in links:
        if l["type"]["name"] != "Blocks":
            continue
        if l.get("inwardIssue", {}).get("key") == bloqueante:
            return True, l["id"]
        if l.get("outwardIssue", {}).get("key") == bloqueante:
            return False, l["id"]
    raise RuntimeError(f"No se encontró el vínculo {bloqueante} -> {dependiente}")


def vincular(bloqueante, dependiente):
    """Crea 'bloqueante blocks dependiente'. La dirección de la API se comprueba con el primer vínculo."""
    global INVERTIR
    if INVERTIR is None:
        _vincular(bloqueante, dependiente, False)
        ok, link_id = _direccion_correcta(bloqueante, dependiente)
        INVERTIR = not ok
        if INVERTIR:
            jira.delete(f"/rest/api/3/issueLink/{link_id}")
            _vincular(bloqueante, dependiente, True)
        print(f"  Dirección de vínculos verificada (invertir={INVERTIR})")
    else:
        _vincular(bloqueante, dependiente, INVERTIR)


def cargar_escenario(escenario):
    proyecto = ESCENARIOS[escenario]["proyecto"]
    if proyecto not in META:
        raise RuntimeError(f"El proyecto {proyecto} no pasó la verificación de la celda 4")
    meta = META[proyecto]
    if not meta["story_points"] or not meta["valor_negocio"]:
        raise RuntimeError(f"{proyecto}: faltan story points o valor de negocio en la pantalla")
    ya = existentes(proyecto, escenario)
    if ya:
        print(f"{escenario}: ya hay {len(ya)} historias con la etiqueta {etiqueta(escenario)} en {proyecto}. "
              "Se omite (usa la celda 11 para eliminarlas y recargar).")
        return None

    filas = BACKLOGS[escenario]
    t0 = time.time()
    claves = {}
    for i, fila in enumerate(filas, 1):
        claves[fila["Issue Id"]] = crear_historia(proyecto, escenario, fila, meta)
        if i % 10 == 0 or i == len(filas):
            print(f"  {escenario}: {i}/{len(filas)} historias creadas")
    n_v = 0
    for fila in filas:
        for d in deps(fila):
            vincular(claves[d], claves[fila["Issue Id"]])
            n_v += 1
    with open(f"escenarios/mapa_claves_{escenario}.json", "w", encoding="utf-8") as f:
        json.dump(claves, f, indent=2)
    print(f"{escenario}: {len(claves)} historias y {n_v} vínculos en {proyecto} "
          f"({time.time() - t0:.0f} s) → escenarios/mapa_claves_{escenario}.json")
    return claves

print("Funciones listas")

Funciones listas


## 9. Cargar los escenarios

In [ ]:
for esc, cfg in ESCENARIOS.items():
    if cfg["cargar"]:
        cargar_escenario(esc)
    else:
        print(f"{esc}: cargar=False, se omite")

E1: cargar=False, se omite
  E2: 10/30 historias creadas
  E2: 20/30 historias creadas
  E2: 30/30 historias creadas
  Dirección de vínculos verificada (invertir=True)
E2: 30 historias y 34 vínculos en TDA (40 s) → escenarios/mapa_claves_E2.json
E3: cargar=False, se omite


## 10. Verificar la carga contra el origen
Vuelve a leer cada escenario desde Jira y lo compara campo por campo con los datos de origen: resumen, story points, valor, prioridad y dependencias. Es la evidencia de que la carga no tiene errores de transcripción.
Las historias se buscan por la etiqueta del escenario; si no la tienen (como E2, cargado antes), se usan todas las del proyecto.

In [ ]:
def leer_de_jira(escenario):
    proyecto = ESCENARIOS[escenario]["proyecto"]
    por_nombre = jira.campos()
    sp = next((por_nombre[n][0] for n in NOMBRES_STORY_POINTS if n in por_nombre), None)
    if proyecto in META and META[proyecto]["story_points"]:
        sp = META[proyecto]["story_points"]
    vn = (META.get(proyecto) or {}).get("valor_negocio") or (por_nombre.get(NOMBRE_VALOR_NEGOCIO) or [None])[0]
    lista = ["summary", "priority", "issuelinks"] + [c for c in (sp, vn) if c]
    issues = jira.buscar(f'project = "{proyecto}" AND labels = "{etiqueta(escenario)}"', lista)
    if not issues:
        issues = jira.buscar(f'project = "{proyecto}"', lista)
    return issues, sp, vn


def verificar_escenario(escenario):
    filas = BACKLOGS[escenario]
    issues, sp, vn = leer_de_jira(escenario)
    por_resumen = {it["fields"]["summary"]: it for it in issues}
    clave_de = {f["Issue Id"]: por_resumen.get(f["Summary"], {}).get("key") for f in filas}
    errores = []
    for f in filas:
        it = por_resumen.get(f["Summary"])
        if not it:
            errores.append(f"Falta en Jira: {f['Summary']}")
            continue
        jf = it["fields"]
        if float(jf.get(sp) or -1) != float(f["Story Points"]):
            errores.append(f"{it['key']}: story points {jf.get(sp)} ≠ {f['Story Points']}")
        if float(jf.get(vn) or -1) != float(f["Valor de negocio"]):
            errores.append(f"{it['key']}: valor {jf.get(vn)} ≠ {f['Valor de negocio']}")
        if (jf.get("priority") or {}).get("name") != f["Priority"]:
            errores.append(f"{it['key']}: prioridad {(jf.get('priority') or {}).get('name')} ≠ {f['Priority']}")
        en_jira = sorted(l["inwardIssue"]["key"] for l in jf.get("issuelinks", [])
                         if l["type"]["name"] == "Blocks" and "inwardIssue" in l)
        esperado = sorted(clave_de[d] for d in deps(f))
        if en_jira != esperado:
            errores.append(f"{it['key']}: depende de {en_jira} ≠ esperado {esperado}")
    extra = len(issues) - len(filas)
    return {"escenario": escenario, "proyecto": ESCENARIOS[escenario]["proyecto"],
            "en_origen": len(filas), "en_jira": len(issues),
            "coincide": "✔" if not errores and extra == 0 else "✘",
            "diferencias": len(errores) + max(extra, 0)}, errores


verificacion = []
for esc in ESCENARIOS:
    r, errores = verificar_escenario(esc)
    verificacion.append(r)
    for e in errores[:10]:
        print(f"  [{esc}] {e}")
    if len(errores) > 10:
        print(f"  [{esc}] … y {len(errores) - 10} más")
tabla_verif = pd.DataFrame(verificacion)
tabla_verif.to_csv("escenarios/verificacion_carga.csv", index=False, encoding="utf-8-sig")
display(tabla_verif)

,escenario,proyecto,en_origen,en_jira,coincide,diferencias
0,E1,BIB,15,15,✔,0
1,E2,TDA,30,30,✔,0
2,E3,SIN,80,80,✔,0


## 11. (Opcional) Restablecer o eliminar un escenario
- **Restablecer:** devuelve todas las historias del escenario al backlog, fuera de cualquier sprint.
- **Eliminar:** borra las historias del escenario para recargarlo desde cero. Pide `CONFIRMAR = True`.

In [ ]:
PROYECTO_A_VACIAR = "TDA"   # clave REAL del proyecto
CONFIRMAR = True           # cambiar a True después de revisar el conteo

issues = jira.buscar(f'project = "{PROYECTO_A_VACIAR}"', ["summary"])
print(f"{PROYECTO_A_VACIAR}: {len(issues)} issues encontradas")
for it in issues[:5]:
    print("  ", it["key"], "-", it["fields"]["summary"])

if CONFIRMAR:
    for it in issues:
        jira.delete(f"/rest/api/3/issue/{it['key']}?deleteSubtasks=true")
    print(f"{len(issues)} issues eliminadas ✔")
else:
    print("Revisa el conteo y pon CONFIRMAR = True para eliminar.")

TDA: 34 issues encontradas
   SCRUM-34 - Registro de auditoría de acciones administrativas
   SCRUM-33 - Diseño adaptable para dispositivos móviles
   SCRUM-32 - Productos recomendados
   SCRUM-31 - Lista de deseos
   SCRUM-30 - Reporte de ventas por período
34 issues eliminadas ✔
